In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
from scipy.optimize import curve_fit

## LOAD 
ARPS_WELLS =['15/9-F-12', '15/9-F-14']  ## only these 2 wells that have clean data to be plotted

print(f"Loading data")
df = pd.read_parquet('../data/processed/cleaned.parquet')
df['DATEPRD'] = pd.to_datetime(df['DATEPRD'])
df = df.set_index('DATEPRD').sort_index()     

df['OIL_RATE_NORM'] = df.groupby('NPD_WELL_BORE_NAME')['OIL_RATE_NORM'].transform(   # first group by names we got 6. then we  take only OILRATENORM coloumn from that.
    lambda x:x.clip(upper=x.quantile(0.99))      # it calculated 99th percentile for this means we are capping the  outliners .
)
print(f'Loaded {len(df)} rows')

## Resampling data to monthly to make it more clean and keep reduce the noise.
print(f"Resampling to month")
monthly_data={}
for well_name , well_df in df.groupby('NPD_WELL_BORE_NAME'):
    monthly = well_df['OIL_RATE_NORM'].resample('ME').mean().dropna()

    if (len(monthly) < 12):
        print(f'Skipping well name: {well_name} as it has less then 12 months data')
        continue

    monthly_data[well_name] = monthly
    print(f'{well_name}: {len(monthly)} months  from {monthly.index.min().date()} --> {monthly.index.max().date()}')


## ARPS CURVE EQUATION
def arps_exponential(t,qi,D):

    ## t : time in months since decline starts
    ## qi: intial Production rate , basically rate when t=0
    ## D: monthly decline rate.
    return  qi * np.exp(-D * t)

def arps_hyperbolic(t,qi,D,b):

    ## t : time in months since decline
    ## qi: intial production rate
    ## D : monthly rate decline , initial
    ## b : Hyperbolic exponential exponent   
    ##  b -> 0 :approch exponential,    b =1 : harmonic decline
    b = np.clip(b,1e-5,0.9999)
    return qi / np.power(1.0 + b * D * t,1.0/b)

def score_forecast(actual,predicted, label=''):

    actual = np.array(actual)
    predicted = np.array(predicted)

    ## RMSE is sqrd root of avg sqrd error    
    rmse = np.sqrt(np.mean((actual-predicted) **2))
    ## MAE is avg of absolute errors
    mae  = np.mean(np.abs(actual - predicted))

    ## R2 is R sqrd, means how much varience is in actual data.
    ss_res = np.sum((actual-predicted) **2)
    ss_tot = np.sum((actual - np.mean(actual)) ** 2)

    r2 = 1 - (ss_res / ss_tot )  if ss_tot > 0 else 0.0
    return {'RMSE': rmse,'MAE' :mae , 'R2' :r2}


print(f'Fitting ARPS DCA on wells: {ARPS_WELLS}')
arps_result = {}

for well_name in ARPS_WELLS:
    if well_name not in monthly_data:
        print(f'{well_name} not in monthly data')
        continue
    print(f"\twell name : {well_name}")
    monthly = monthly_data[well_name]
    ## we find peak first and then decline starts as ARPS DCA build decline curve

    ## we igonre build up in curve as it distorts D paramater
    ## so we have production Peak and then fit on that.
    peak_idx = monthly.idxmax()
    peak_pos = monthly.index.get_loc(peak_idx)
    decline_d = monthly.iloc[peak_pos:]

    print(f'Peak {peak_idx.date()} {monthly.max()} sm^3/DAy')
    print(f'DEcline period {len(decline_d)} month')

    # TRain / TEST  split 
    n_train = int(len(decline_d) * 0.8)
    train = decline_d.iloc[:n_train]
    test = decline_d.iloc[n_train:]

    print(f'Train:{len(train)} months | {len(test)} months')

    ## we arrange in array starting from peak of slope
    T_all = np.arange(len(decline_d), dtype=float)
    T_train = T_all[:n_train]
    T_test = T_all[n_train:]


    qi_guess = float(train.iloc[0])        ##  first training value , intial production guess
    D_guess = 0.05            ## 5% monthly decline

    try:
        popt_exp,_ = curve_fit(       ## curve fit tries to find best qi, D
            arps_exponential,
            T_train,
            train.values,
            p0=[qi_guess,D_guess],
            bounds=([0,0.001],[qi_guess*2 , 0.5]),     ##  Lower bound and upper bound, limits
            maxfev=10000
        )
        qi_exp, D_exp =popt_exp  ## Extract the fitted paramters
        forecast_exp_train = arps_exponential(T_train, *popt_exp)   ## predict training data
        forecast_exp_test = arps_exponential(T_test,*popt_exp)  ## predict future data
        score_exp = score_forecast(test.values, forecast_exp_test) ## forcast using RMSE, MAR, R^2
        exp_ok = True
        print(f'Exponential Fit: qi = {qi_exp:0f}  D ={D_exp:0f} per month {D_exp*100} /month')
        print(f'exponential test : RMSE :{score_exp['RMSE']:1f}   MAE = {score_exp['MAE']:1f}  R^2 = {score_exp['R2']:3f}')
    except Exception as e:
        print(f'Exponential Fit failed')
        exp_ok = False
        score_exp = {'RMSE' : 1e9,'MAE':1e9 , 'R2':1e9}

    try:
        popt_hyp,_ = curve_fit(
            arps_hyperbolic,  ## mathemactical function to fit to data
            T_train,
            train.values,
            p0=[qi_guess,D_guess,0.5],
            bounds=([0,0.001,0.01],[qi_guess*2,0.5,0.99]),
            maxfev=10000
        )
        qi_hyp , D_hyp,b_hyp = popt_hyp
        forecast_hyp_train = arps_hyperbolic(T_train,*popt_hyp) ## * to unpack this list
        forecast_hyp_test = arps_hyperbolic(test.values,*popt_hyp)
        score_hyp = score_forecast(test.values, forecast_hyp_test)
        hyp_ok =True
        print(f'hyperbolic fit is , qi = {qi_hyp:0f}  D={D_hyp:0f}  , {b_hyp:3f}')
        print(f'Hyperbolic test is : RMSE :{score_hyp['RMSE']:1f}   MAE = {score_hyp['MAE']:1f}  R^2 = {score_hyp['R2']:3f}')
    except Exception as e:
        print(f'hyperbolic fit failed')
        hyp_ok = False
        score_hyp = {'RMSE' : 1e9,'MAE':1e9 , 'R2':1e9}

    
    if exp_ok and hyp_ok:
        winner = 'exponential' if score_exp['RMSE'] < score_hyp['RMSE'] else 'hyperbolic'
    elif exp_ok:
        winner = 'exponential'
    elif hyp_ok:
        winner = 'hyperbolic'
    else:
        print(f'Both curve fit fails for well : {well_name}')
        continue
    
    print(f' Best model : {winner}')

    arps_result[well_name]={
        'monthly' :     monthly,
        'decline_d':    decline_d,
        'train':        train,
        'test':         test,
        'T_train':      T_train,
        'T_test' :      T_test,
        'T_all':        T_all,
        'peak_idx':     peak_idx,
        'winner' :      winner,
        
        'exp_ok' :      exp_ok,
        'popt_exp':     popt_exp,
        'forecast_exp_test':   forecast_exp_test if exp_ok else None,
        'forecast_exp_train': forecast_exp_train if exp_ok else None,
        'score_exp':           score_exp,

        'hyp_ok':       hyp_ok,
        'popt_hyp':     popt_hyp,
        'forecast_hyp_test': forecast_hyp_test,
        'forecast_hyp_train': forecast_hyp_train,
        'score_hyp':        score_hyp,
    }
    


Loading data
Loaded 8020 rows
Resampling to month
15/9-F-1 C: 25 months  from 2014-04-30 --> 2016-04-30
15/9-F-11: 39 months  from 2013-07-31 --> 2016-09-30
15/9-F-12: 102 months  from 2008-02-29 --> 2016-08-31
15/9-F-14: 97 months  from 2008-07-31 --> 2016-07-31
15/9-F-15 D: 30 months  from 2014-01-31 --> 2016-07-31
Skipping well name: 15/9-F-5 as it has less then 12 months data
Fitting ARPS DCA on wells: ['15/9-F-12', '15/9-F-14']
	well name : 15/9-F-12
Peak 2008-12-31 5365.325161290323 sm^3/DAy
DEcline period 92 month
Train:73 months | 19 months
Exponential Fit: qi = 5801.036501  D =0.046185 per month 4.618478046423671 /month
exponential test : RMSE :312.716346   MAE = 265.984038  R^2 = -1.497675
hyperbolic fit is , qi = 5809.428053  D=0.046563  , 0.010000
Hyperbolic test is : RMSE :447.281061   MAE = 399.922125  R^2 = -4.109704
 Best model : exponential
	well name : 15/9-F-14
Peak 2008-10-31 4504.44341175957 sm^3/DAy
DEcline period 94 month
Train:75 months | 19 months
Exponential F